below is a data processing step in *cleanData()* we load all the files then drop all the empty and duplicate entry. Next, we change all ids into string so there will be no type accident in calcuation

In *groupData()* we created a few variables to use in calculation

*train_user_watched* is a dict of what user have watched in training set

*val_user_movie* is a dict of validation set data which is used as a ground truth to evaluate the presicion of the model

*all_movies* is all of the movies from the training set

based on *train_user_watched* the function *get_candidate_movies()* is used to get all of the movies that user have not seen for a potential suggestion

*allData()* is function that have all processed data to make calling each variable easier


In [ ]:
import pandas as pd

def cleanData():

    train = pd.read_csv("train.csv")
    val = pd.read_csv("val.csv")

    movies = pd.read_csv(
        "movies.dat",
        sep="::",
        engine="python",
        names=["movie_id", "title", "genres"],
        encoding="latin-1"
    )

    users = pd.read_csv(
        "users.dat",
        sep="::",
        engine="python",
        names=["user_id", "gender", "age", "occupation", "zip"],
        encoding="latin-1"
    )

    train = train.drop(columns=["timestamp"]).dropna().drop_duplicates()
    val = val.drop(columns=["timestamp"]).dropna()

    train["user_id"] = train["user_id"].astype(str)
    train["movie_id"] = train["movie_id"].astype(str)

    val["user_id"] = val["user_id"].astype(str)
    val["movie_id"] = val["movie_id"].astype(str)

    movies["movie_id"] = movies["movie_id"].astype(str)
    users["user_id"] = users["user_id"].astype(str)

    return train, val, movies, users

def groupData(train, val):
    train_user_watched = train.groupby("user_id")["movie_id"].apply(set).to_dict()

    val_user_movie = val.set_index("user_id")["movie_id"].to_dict()

    all_movies = train["movie_id"].unique()

    return train_user_watched, val_user_movie, all_movies

def get_candidate_movies(user_id, train_user_watched, all_movies):
    seen = train_user_watched[user_id]
    return [m for m in all_movies if m not in seen]

def allData():
    train, val, movies, users = cleanData()

    train_user_watched, val_user_movie, all_movies = groupData(train, val)

    return {
        "train": train,
        "val": val,
        "movies": movies,
        "users": users,
        "train_user_watched": train_user_watched,
        "val_user_movie": val_user_movie,
        "all_movies": all_movies
    }

